# RetinAI CNN v2 — revisión de errores

Esta libreta **no entrena**. Calcula predicciones de los dos modelos ya guardados sobre las mismas 673 imágenes de validación ODIR y crea láminas de ejemplos confundidos con «normal». Sirve para revisar imágenes y anotaciones antes de modificar otra vez el entrenamiento. Puede ejecutarse sin GPU.

Ejecuta todas las celdas y comparte `retinai_errores_validacion.zip`.


In [ ]:
%pip -q install kagglehub pillow


In [ ]:
import base64
import gzip
import io
from pathlib import Path
import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

CLASES = ["normal", "retinopatia_diabetica", "glaucoma", "catarata"]
DATOS_B64 = "H4sIACmz7moC/32c244dtxFF3w34TwZB805+jTCRFXkCW1IU2d8fsgOo1xqAfNBD6Zzp003WZdeuzX79/vH3t7+/vrz99uHb68e3T19+fHr59OPtP399+vH64dv3r3+//fft65fXP379Jff84fvb599//OPf3z6/TOvl+/zil6/fXn+8vX747e31n9P8+PrrL+ka48Mfn/71/y8u6+XL1+9/rmvENsrz0bKej3q/cP1l/vwshx7529N8Prui7muauxuLJfEXSnouEkvhRaa5vUjLlY+Q63OVcvERlrldo9Eb1mhau2/m3PTs09x+NZb4XHRZ24fIAz+/rJ8PMfeL67nMZ4/K1bSC1/N3sVZeclo/Pwpalmk9K5ayHm6a21vuOcg/cnh+O2b5zjRfPr7+eP0+/80nig1Otyz8/qj6/VG3Oxblg8v8eZka4PDT2EdGzYyMmunGyW6c9ndy+U6u505iol8t69nYyD1fFva8dd5X6/DppsiY5sH9utyvb7+5Hvb55nrWJxIHP5rW7iK90U+ntftiiFyTZb18/uP1r49f/1wfhoSYWRZvRXfyLFfWXi0TqyxXXCZSXGT2m9bz0QiKrGlid6p+bpo/P8P1yj7XFPr/2Oe2ubtOYO3ap8GkcJwmo8pRPbZRnVLVck3zee4Uqz6LT6od5XoeaRqoMPO+kJ5xW/PJq9ah7v2zuaa0J1BTGPjpZcFhovJv3Cez1jPvsmd6nS4yIvJoyEqkAak558SMnh8fieFS3ZsmsmMN/HBaz53MWqPKk9thG3HTy8LzhK7SGrZpIYbmG2280RyiMtE0dxdqghINSGJkpIKRD1WXvrKs7U1H3dYyWXuGMMcyD7DJYT6QVVJpCoWCwltKVFEuz+KXwT+b1t4ha1c41w6PVzQsk2jhEtZDJIbAbDet/ZMnlZ+E8hO0KMvcAwRlUFa+UXmT0zoUeUGFDqiQlfCWub2TlAdvZZqoAUxPfX+NkHSNadKtrq7UOE3caGMFndb2aWcG0cZeT7KclbCqLu6TZWhDW9TGfovsZb0SaNCTlrXfpCAgg1I5r3hxeVlNLgILRVeeG616FeH8UfhH6LEQ8U5jC0ByxqYvCwAkpdYV2gRflUmo7nPQCjU6S2AO6rzKNHB5Xn3fKziwMiJrXNz6acELlazzCXMU7vseX8aUWOKmxf7NRWaPPROfOqF7G4FpdFrA5llwNW8jKoSqqMFiz8JAZD4tbNFMEd0Z43GCllTM0j4ukjqkZaJIVHdulUFTBoOmYBdjU9Kb5qErUeq8CnKJrrLMJ4YIsgtA9qzv1eV+n4ESH2BZ+4wnKDwt9K+RkGxaTw4oqn8lY+eKGsiy7x9jScXsA2vLMNI6FOqgzZoWIVJVJq5jn7MZ+gFeumKMjsgg67p6P7S5F9Hghf40RbZby2Kyu5Tt9r2Hn3OZcPROeqawMKZGZJ0OREpgZl8WLlLMGByQ4Swkuk08bOhDnEgncOpN8diGyS42znBS459EADQdigEyrWM/RiCP21pRRtdAnpg1TY8zTZS4OKNJELnGI+5W3ThEfdZ27qmjEIkvl7UP0kJ4Ni3kI4GlLLB0Dab9aRGNaen34CgUtWnL5BpmETsswrFevOmKG7sJJ1EH6BijOr9lMpVEOegyEWPZRCqbzTFUyaaJuxEoWCY8u8mzD3ROqrzKMgmnrksNCcjGlvAL03jou4g/mYZrAh6GReF2FbVd8IeVoAD4Zn7CHQ5THgMxdoNgEjBiF0SMoTkcXtZxAFsxNV3k0NFnd5xZhFS3V3GPkz9DM+dP8P+jqT9rTDrRSYcIIZkyTXvKNF4sO8vag0zVVya5mYvUoRwKVGuaEbRqqpXr09hrjKEUzslJ5PYvi8FaruzBw344k4NA4jSB04YiKwxBlKgSsl/B0dmDjc7EwUhbFvgr9VnLROoVeL3r8RZ6sPyrLw4ukO944MwsfcBNnauw5lDb+HH4kIe/qul7ekdS9kpcu0Q6hfAxFLV/y7RPkfUXjU0sNi17m0gtultsWWlxmrgX3CXxdUpiipKYIo1WlqneyIk2IdHG1s1Cd3QWxOX7hCjKcZmHLqOqy9iPirKmWssEyiUDuayDByW7UKIzmPMjtFwRjuITgqBYDN7YgKUe9JVpMMWIX17WAfYGw95gaCuSjr1mFfG9B4mx8pvLQsUurtiFCDkYIYM4uikaAVikxWjafZpc0V49tybnFFTONSyITsSRiXgio8A0wEa+Gkvz4bNryzK3zsUClVif1Poktj7RXaqmxtlBmhmk0/3Fpe8b+Xt6yVAniTvvReGArlWdNfvqIPB9YCY7k2vHJe5xv3JdNXHCrSdYyFFIbpp7UoLQV/T6PQwSCj8EhSN7ms+NquKsgnMicVSbOCvnzG1Z28WUi3aG55BIYMC7Zv1pHqrC81IWKAOfPkt8NBMePTjluD1xpmHqG5SUwPuBREmhO7qBdEK8DDaolFgjYkA5iRVCILm9LHMn3JzER7UmgJxLJ2cwDVXy7kreWclZyHU1bC4ZiHwJ2k4T+TVV5ddUmUMLH3pZmGBl8WsH1ZFZuguZ/0buxMcH6B41o/c0L1dVkGXuMQCJzST5QNC4K6ATitkZJRd6epY35+TZFDLsfjTVWOHbngeaRY6yiGkRKweu0WkQe2dBpv79uo+hroGVMlXx/gjjmN0ecpkbo6xRhtE0gyNmvMeZmsE+f9Yv7No09jjxEi91nVRY3QW1c/iePHzfA9MwxOYeNExrjAwMXpq9S9xMwpQHCWTse8tb3cdsov5RuPeoJSBckOojuREUOkmj6bNpPq7AHWnXO3ECMGfoVJ8YfpNxWbkGoSiQycVaFkdihgTlwHATwwsbzhjyqGvvibUx507rgB2KsUNxySahxzVPqvTvdB9DSwvfvkdhakKayAyK0ZZFAZWxdiTWjsqqmuFKbElmPCpNK39MKGG2EmKA2QKobk5TZYJs07L21caD24hCm42oSzgmHGJvOPidx9W7MJHf+86ZF5/QE2V+cvnZr2zBi8Ow7MfiwZjgoDGIoXZOvWrncifu/D293/q5cEYFzsiWvS7zqL9kiEezZBrk7Qmizmre00EUZSFIY1t+EwFkhbD1hNorUxJvUSKyLKdRcrHtKIhlVD0/PSQYGmhootR5BwlNLxJyFMoCDfQSgV4Wg3qadOVkzQewqnH2MrF2TQE1reMjqDu8rBLF2uXLrDa9ee8ZlS60LKto1TceaC0NXpK6T41Bo8agVqUfhqk38cf94kwjuSQemsp6ARVUcqaZwExdx+whs9m9zKMIXUcRuo8iwPk6nU/MjCQWrZspAPd8c75MGZSdUJEY3un6OIuMVLJ64rOX9IagwWxAi3qp11gmQoBTkGUxewyin2U9GFmDoSXv2kefx+j9rO6Qj+z1ZVnDG3dPxXOXAnwg8SowzM05YbcplGjdgsneSVeoYeF4tFus2Uu0CoQjloMIpEla2dik3nNkdO5X8uAWLoUbM7J2uQiWQwfJod2BqdGdmCUZwhzEr1US0NqtLaB6jtKCW+yo0zHp0J14xziZHsJzy9zr1USDlX3GuiEFRtp7RHF3cFK5HqC60Vdlg5G6JhT9IEZyrqJMJopMEH1/Q2JO18QOFrGDxW6AFdt7QUgqW8vkXieNINPgVEUl+wS5k+boPruRCIeXReFNk/CGtFwRDkJE34M5hGKgUuFu5FBdLx266T6H0Y8DLFzloPbNfATxq4VpsxROnTVpWTMZkYeqoL27fhBCqYAkBYZOmCU1vppO56KucZnP7yWKe2+/0bk0DUzwZ6EWyTXR5RQfyCEnmYuVcNP0iTI66UEexuISqcJLPgyWeEBK5eOYJSgJyvZkHTTbI/whsd7YB1XT9Kzl4HknI3NP/MQh6d4gfhqShZ3O7sQgXDctdqeL9ueohQpZD0kKIWYS38OOI/Cewz7jdvaoneTEmsKrgEYolpKkvwclv7U14pRTtaST/Gkn+F8WeB+fx6WOIyRp4ZapmtwleetEYVa48GywqjPR+FIb8MAtGxSpLG49k/TDJKnZMWQJHzLnx12LpaaqXKr8lJZRwNSgX7pP71L+dxAeWVRH1Hj3aOq28pEuIDTUEN8qeUDgQIp9Gvtmjkddl+Uxryae1QovkbN71PROZnzQGd/iMOBsacN8oDvtO6TOsthZVLKOwHhkdctB4V7XfuRCirntT78N5Z9xVZ/t45yADUWTpKrFy3wzz7BdB8G2w5IHOapnZ5VyRQ8NTicjbjULHIdiFk/VRMv5WOzYz9viuxn3arF5Xkqg75J6UjORA18Z6/CEfQScAO86RUlBZPFklh1fotz4Fk2cjmqhcF3BLSWpMp4G13God2puMXEUVRUfwyQ9mE3dL/PQ1Pi85tiGYBVDVPdHv+4DZDzf9axzkU6jHM4ziYt55yg3yESOjwe0Ii5ZzE2k0nhZbM+j6uoyPd/gAXHKhaqoDR150PJFiobvsark2Jel6eqdt5m2kGmchoWTuMbwiASFSBOS+0USICIuguvqIrUnkYag5eE07y0SYd8q0sUEwji8d+GdOhdJOFhGgYPZgy3gkDJTlGCFJw95Ms9EcR+Wtc9V4ugkgwvRWDG+S1YcbFJWKKHNMo8gQVKt4e2liHovUfMhIrGfSRl7zydUs581XD4RBxckuR3ZJdyH3tCZCwM26fPVzxuCX4bgl6MX94EALcKiDrv7JBt3kYRLFBdzeJlN0HGsd5OO+1wB1jkfDnDrmzzXwQLEp+5Z56DZMRb2rict+03FMVUcEE50FSZLk6uOD9YDvxxMIbq5rPL6w2tg7hMsYCuqaSHvnQ+VMAB1qmSGeLfwFe6n9+/EiwdcugbAnXptvROGx5JvrT5qEaX6Rb1f2YP37B05va3oztEkxtNRBMmBdqB6teuVR51vKvC2hm6aS4c5dNS+CGAf+XrNcIt7aTiDWmnzDiU72ajzPTCEPj0R62GuwDr5Tofv8VbbUx33IQYdAXocrkqRotPRQbI7ytKD68kyFTCCbCdEqvH/Sb2nLT88agi+sxAICprngVRZx2EpOqXUMTowYjwyZuJoqmUa2EupNG7uUeLz6o6S902RTbQ8uFjuo4nz4e1PEvgcVOr3yUYNO/jOrygYH9XvpSyhW+bLyHQYVOfThgW2B8Lzbm6ouTvUxsvRwNefRc3e457WaTpfu9SM23dNMBrCOxV111hSulPNTT02JSAkL6deTG+MGcT+p2naEB/oIye3OAo7rJda3OJrNMTsooIbD79MhpuWjKz1hsHDHFHXJ9ZvHHS1w7QzWlrG98w0TgUahwIShrzTI/rNDzwwcnM7PBA7fFaXmls6hHRFPLp+E+Xs3arhhNhTIoMR9H6XcHw3IsNw/3LEofQ7AkXWPvkccPI5JZ8kqXpf0tPnHgbwlWFhsNWtpdRb3vzOOvHt/wPAwAYriVMAAA=="
tabla = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(DATOS_B64))), dtype={"id_paciente": str})
assert len(tabla) == 673 and tabla["archivo"].is_unique
tabla = tabla.sort_values("archivo").reset_index(drop=True)
tabla["clase_id"] = tabla["etiqueta_provisional"].map({c: i for i, c in enumerate(CLASES)})
raiz = Path(kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k"))
carpetas = [p for p in raiz.rglob("Training Images") if p.is_dir()]
if len(carpetas) != 1:
    raise RuntimeError(f"Carpetas Training Images encontradas: {carpetas}")
tabla["ruta"] = tabla["archivo"].map(lambda x: str(carpetas[0] / x))
if not tabla["ruta"].map(lambda x: Path(x).is_file()).all():
    raise RuntimeError("Faltan imágenes de validación")
print("Imágenes:", len(tabla))


In [ ]:
def abrir(ruta, clase):
    imagen = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    imagen = tf.image.convert_image_dtype(imagen, tf.float32)
    imagen = tf.image.resize_with_pad(imagen, 224, 224)
    return imagen * 255.0, clase

ds = tf.data.Dataset.from_tensor_slices((
    tabla["ruta"].to_numpy(), tabla["clase_id"].to_numpy(dtype="int32")
)).map(abrir, num_parallel_calls=tf.data.AUTOTUNE).batch(24).prefetch(tf.data.AUTOTUNE)

from google.colab import drive
drive.mount("/content/drive")
destino = Path("/content/drive/MyDrive/RETINAI_CNN_V2")
rutas_modelos = {
    "base": destino / "clasificador_odir_4clases_exploratorio.keras",
    "nuevo": destino / "clasificador_odir_sylhet_4clases_exploratorio.keras",
}
if any(not ruta.is_file() for ruta in rutas_modelos.values()):
    raise RuntimeError(f"Faltan archivos de modelo: {rutas_modelos}")
probabilidades = {}
for nombre, ruta in rutas_modelos.items():
    print("Evaluando:", nombre)
    probabilidades[nombre] = keras.models.load_model(ruta).predict(ds, verbose=1)
    tabla[f"pred_{nombre}"] = [CLASES[i] for i in probabilidades[nombre].argmax(axis=1)]
    tabla[f"conf_{nombre}"] = probabilidades[nombre].max(axis=1).round(4)
    for i, clase in enumerate(CLASES):
        tabla[f"p_{clase}_{nombre}"] = probabilidades[nombre][:, i].round(4)

salida_csv = Path("/content/retinai_errores_validacion.csv")
tabla.drop(columns=["ruta", "clase_id"]).to_csv(salida_csv, index=False)
print("Errores de retinopatía como normal:", int((tabla["etiqueta_provisional"].eq("retinopatia_diabetica") & tabla["pred_nuevo"].eq("normal")).sum()))
print("Errores de glaucoma como normal:", int((tabla["etiqueta_provisional"].eq("glaucoma") & tabla["pred_nuevo"].eq("normal")).sum()))


In [ ]:
from PIL import Image, ImageDraw, ImageOps
from zipfile import ZipFile, ZIP_DEFLATED

def lamina(filas, titulo, salida, limite=24):
    filas = filas.head(limite)
    columnas = 4
    ancho_celda, alto_celda = 260, 270
    filas_n = max(1, (len(filas) + columnas - 1) // columnas)
    lienzo = Image.new("RGB", (columnas * ancho_celda, 45 + filas_n * alto_celda), "white")
    dibujo = ImageDraw.Draw(lienzo)
    dibujo.text((10, 10), f"{titulo} (n={len(filas)})", fill="black")
    for pos, (_, fila) in enumerate(filas.iterrows()):
        x = (pos % columnas) * ancho_celda
        y = 45 + (pos // columnas) * alto_celda
        with Image.open(fila["ruta"]) as original:
            imagen = ImageOps.contain(original.convert("RGB"), (248, 218))
            lienzo.paste(imagen, (x + (ancho_celda - imagen.width) // 2, y))
        dibujo.text((x + 5, y + 220), str(fila["archivo"]), fill="black")
        dibujo.text((x + 5, y + 237), f"Real:{fila['etiqueta_provisional'][:6]} B:{fila['pred_base'][:6]} N:{fila['pred_nuevo'][:6]}", fill="black")
        dibujo.text((x + 5, y + 252), f"P(normal) nuevo: {fila['p_normal_nuevo']:.2f}", fill="black")
    lienzo.save(salida)

carpeta = Path("/content/retinai_laminas")
carpeta.mkdir(exist_ok=True)
dr = tabla.loc[
    tabla["etiqueta_provisional"].eq("retinopatia_diabetica") &
    tabla["pred_nuevo"].eq("normal")
].sort_values("p_normal_nuevo", ascending=False)
glaucoma = tabla.loc[
    tabla["etiqueta_provisional"].eq("glaucoma") &
    tabla["pred_nuevo"].eq("normal")
].sort_values("p_normal_nuevo", ascending=False)
aciertos_glaucoma = tabla.loc[
    tabla["etiqueta_provisional"].eq("glaucoma") &
    tabla["pred_nuevo"].eq("glaucoma")
].sort_values("p_glaucoma_nuevo", ascending=False)

lamina(dr, "Retinopatía confundida con normal", carpeta / "retinopatia_como_normal.png")
lamina(glaucoma, "Glaucoma confundido con normal", carpeta / "glaucoma_como_normal.png")
lamina(aciertos_glaucoma, "Glaucoma reconocido", carpeta / "glaucoma_reconocido.png")

paquete = Path("/content/retinai_errores_validacion.zip")
with ZipFile(paquete, "w", ZIP_DEFLATED) as zipf:
    zipf.write(salida_csv, arcname=salida_csv.name)
    for imagen in carpeta.glob("*.png"):
        zipf.write(imagen, arcname=imagen.name)
print("Paquete:", paquete)


In [ ]:
from google.colab import files
files.download(str(paquete))
